# AlphaEarth Montpellier - `mtp_points`

Ce notebook prépare la grille de points à 10 m sur Montpellier, découpe cette grille en lots pour Earth Engine, puis fusionne les exports AlphaEarth.

Sortie finale attendue : `data/processed/alphaearth/alphaearth_data/mtp_points.csv`.


In [ ]:
from pathlib import Path
import math
import numpy as np
import pandas as pd

ROOT = Path.cwd()
AE = Path("data/processed/alphaearth/alphaearth_data")
EXPORT_DIR = Path("data/external/alphaearth/mtp_alphaearth")
AE.mkdir(parents=True, exist_ok=True)
EXPORT_DIR.mkdir(parents=True, exist_ok=True)

AE_YEAR = 2024
GRID_SPACING_M = 10
CHUNK_SIZE = 25_000

print("AE ->", AE)
print("EXPORT_DIR ->", EXPORT_DIR)


## 1. Grille à 10 m

La grille est relue si `data/processed/alphaearth/alphaearth_data/mtp_grid_10m_points.csv` existe déjà. Sinon, elle est reconstruite à partir d'une emprise large de Montpellier.


In [ ]:
BBOX_MONTPELLIER = {
    "lon_min": 3.79,
    "lat_min": 43.56,
    "lon_max": 3.96,
    "lat_max": 43.67,
}


def build_regular_grid(bbox, spacing_m=10):
    lat_mid = (bbox["lat_min"] + bbox["lat_max"]) / 2
    lat_step = spacing_m / 111_320
    lon_step = spacing_m / (111_320 * math.cos(math.radians(lat_mid)))

    lats = np.arange(bbox["lat_min"], bbox["lat_max"] + lat_step, lat_step)
    lons = np.arange(bbox["lon_min"], bbox["lon_max"] + lon_step, lon_step)
    lon_grid, lat_grid = np.meshgrid(lons, lats)

    grid = pd.DataFrame({
        "uid": np.arange(lon_grid.size, dtype="int64"),
        "longitude": lon_grid.ravel(),
        "latitude": lat_grid.ravel(),
    })
    return grid


grid_path = AE / "mtp_grid_10m_points.csv"

if grid_path.exists():
    grid = pd.read_csv(grid_path)
    print("Grille existante relue :", grid_path)
else:
    grid = build_regular_grid(BBOX_MONTPELLIER, GRID_SPACING_M)
    grid.to_csv(grid_path, index=False)
    print("Grille créée :", grid_path)

print("Nombre de points :", len(grid))
display(grid.head())


## 2. Découpage en lots

Les lots de 25 000 points facilitent l'upload manuel dans Earth Engine. Le manifest garde le chemin du CSV et l'Asset ID à utiliser.


In [ ]:
chunk_dir = AE / "mtp_grid_10m_chunks_25k"
chunk_dir.mkdir(parents=True, exist_ok=True)

asset_prefix = "projects/ee-acombesaguera/assets/mtp_points_chunks/mtp_points_uid"
rows = []

for start in range(0, len(grid), CHUNK_SIZE):
    stop = min(start + CHUNK_SIZE, len(grid))
    chunk = grid.iloc[start:stop].copy()
    first_uid = int(chunk["uid"].min())
    last_uid = int(chunk["uid"].max())
    csv_path = chunk_dir / f"mtp_points_uid_{first_uid}_{last_uid}.csv"
    chunk.to_csv(csv_path, index=False)
    rows.append({
        "chunk_id": len(rows),
        "start_uid": first_uid,
        "end_uid": last_uid,
        "n_points": len(chunk),
        "csv_path": str(csv_path),
        "asset_id": f"{asset_prefix}_{first_uid}_{last_uid}",
    })

manifest = pd.DataFrame(rows)
manifest_path = AE / "mtp_grid_10m_chunks_25k_manifest.csv"
manifest.to_csv(manifest_path, index=False)

print("Grille :", grid_path)
print("Manifest :", manifest_path)
display(manifest.head())


## 3. Upload manuel des lots

1. Créer le dossier `mtp_points_chunks` dans `projects/ee-acombesaguera/assets`.
2. Aller dans l'onglet `Assets` de Google Earth Engine.
3. Sélectionner un CSV du dossier `data/processed/alphaearth/alphaearth_data/mtp_grid_10m_chunks_25k/`.
4. Mettre comme Asset ID le chemin indiqué dans le manifest.
5. Vérifier `longitude` comme X et `latitude` comme Y si l'interface le demande.
6. Lancer l'upload et attendre la fin de la tâche.


In [ ]:
manifest = pd.read_csv(AE / "mtp_grid_10m_chunks_25k_manifest.csv")
manual_upload_list = manifest[["csv_path", "asset_id"]]

if manual_upload_list.empty:
    print("Aucun lot à uploader.")
else:
    first = manual_upload_list.iloc[0]
    print("Premier fichier à uploader :")
    print(first["csv_path"])
    print("Asset ID à utiliser :")
    print(first["asset_id"])

print("Liste complète :", manual_upload_list)
display(manual_upload_list.head())


## 4. Export AlphaEarth depuis les assets uploadés

Après l'upload manuel du premier lot, lancer `export_asset_chunks(max_chunks=1)` pour tester. Quand le test est validé, relancer sans limite.


In [ ]:
def alphaearth_image(year=AE_YEAR):
    import ee

    collection = ee.ImageCollection("GOOGLE/SATELLITE_EMBEDDING/V1/ANNUAL")
    return collection.filterDate(f"{year}-01-01", f"{year + 1}-01-01").mosaic()


def export_asset_chunks(max_chunks=None, drive_folder="MTP_alphaearth"):
    import ee

    ee.Initialize()
    manifest = pd.read_csv(AE / "mtp_grid_10m_chunks_25k_manifest.csv")
    if max_chunks is not None:
        manifest = manifest.head(max_chunks)

    image = alphaearth_image(AE_YEAR)
    bands = [f"A{i:02d}" for i in range(64)]
    tasks = []

    for _, row in manifest.iterrows():
        asset_id = row["asset_id"]
        export_prefix = f"PICOPATT_AEF_points_{AE_YEAR}_uid_{int(row['start_uid'])}_{int(row['end_uid'])}"
        points = ee.FeatureCollection(asset_id)
        samples = image.select(bands).sampleRegions(
            collection=points,
            properties=["uid", "longitude", "latitude"],
            scale=10,
            geometries=False,
        )
        task = ee.batch.Export.table.toDrive(
            collection=samples,
            description=export_prefix,
            folder=drive_folder,
            fileNamePrefix=export_prefix,
            fileFormat="CSV",
        )
        task.start()
        tasks.append(task)
        print("Export lancé :", export_prefix, task.id)

    return tasks


# Test après upload manuel du premier asset.
# tasks = export_asset_chunks(max_chunks=1)

# Lots restants après validation du test.
# tasks = export_asset_chunks()


## 5. Fusion des exports téléchargés

Les exports AlphaEarth sont lus depuis `data/external/alphaearth/mtp_alphaearth/`, puis fusionnés dans `data/processed/alphaearth/alphaearth_data/mtp_points.csv`.


In [ ]:
export_dirs = [EXPORT_DIR]
export_files = []
for directory in export_dirs:
    if directory.exists():
        export_files.extend(sorted(directory.glob("*.csv")))

if not export_files:
    print("Aucun export trouvé.")
else:
    frames = []
    for path in export_files:
        df = pd.read_csv(path)
        df["source_file"] = path.name
        frames.append(df)

    mtp_points = pd.concat(frames, ignore_index=True)
    drop_cols = [c for c in mtp_points.columns if c.startswith("system:") or c == ".geo"]
    if drop_cols:
        mtp_points = mtp_points.drop(columns=drop_cols)

    if "uid" in mtp_points.columns:
        mtp_points = mtp_points.drop_duplicates(subset=["uid"]).sort_values("uid")

    out_path = AE / "mtp_points.csv"
    mtp_points.to_csv(out_path, index=False)
    print("Fichier fusionné :", out_path)
    print("Shape :", mtp_points.shape)
    display(mtp_points.head())
